"""
SID - Système d'Information Décisionnel
Région Béni Mellal-Khénifra : Prix des Matériaux de Construction (2005-2019)
============================================================================
Script Python complet : Nettoyage, Analyse, KPI, ETL Automatisé
============================================================================
"""

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================
DATA_PATH = "./data/prix_materiaux_construction_beni_mellal_khenifra.csv"
DB_PATH   = "./data/materiaux_beni_mellal.db"
YEARS     = [str(y) for y in range(2005, 2020)]

# Chargé, nettoyé le fichier excel, et préparé les données pour l'analyse.


In [ ]:
# Charger le fichier Excel
df = pd.read_excel("./data/prix-et-indices-des-materiaux-construction-region-benimellal-khenifra.xlsx")

# Supprimer la première ligne
df = df[2:].reset_index(drop=True)

# Renommer les colonnes
df.columns = ["corps", "activite", "produit", "variete"] + YEARS

# Propager les valeurs manquantes dans les colonnes "corps", "activite", "produit"
df[["corps", "activite", "produit"]] = df[["corps", "activite", "produit"]].ffill()

# Supprimer les valeurs dupliquées
df = df.drop_duplicates().reset_index(drop=True)

# Sauvegarder le DataFrame nettoyé dans un nouveau fichier CSV
df.to_csv(DATA_PATH, index=False)

print(f"Fichier nettoyé sauvegardé sous: {DATA_PATH}")
print("Aperçu des données nettoyées:")
df.head()

Fichier nettoyé sauvegardé sous: prix_materiaux_construction_beni_mellal_khenifra.csv
Aperçu des données nettoyées:


,corps,activite,produit,variete,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
0,Gros œuvre,Agglomérés et articles en ciment ou en PVC,Agglomérés,15-20/20/40-50 (u),3.234704,3.234704,3.234704,3.459972,3.655227,3.605077,3.795395,4.055748,4.306207,4.546307,4.865043,4.891272,4.919565,4.898861,5.061590
1,Gros œuvre,Agglomérés et articles en ciment ou en PVC,Agglomérés,"7,5-10 /20/40-50 (u)",2.265125,2.265125,2.265125,2.461798,2.543180,2.797498,2.950088,2.970434,2.797498,2.848361,2.922466,2.945012,2.979100,2.917660,3.066417
2,Gros œuvre,Agglomérés et articles en ciment ou en PVC,Hourdis,12-16/20/52 (u),3.914878,3.914878,3.914878,4.208494,4.110622,4.453174,4.697854,4.844662,4.991470,5.236149,5.568657,5.610389,5.580810,5.586330,5.586330
3,Gros œuvre,Agglomérés et articles en ciment ou en PVC,Hourdis,20/20/52 (u),4.98077,4.980770,5.478847,5.976924,5.976924,6.475001,6.624424,6.544732,6.475001,6.574616,6.638590,6.706429,6.667415,6.373800,6.373800
4,Gros œuvre,Agglomérés et articles en ciment ou en PVC,Buse en béton ou PVC,D 100 à 150 mm (ml),18.18147,19.250968,19.785718,20.855216,21.924714,20.320467,21.817764,21.122590,19.464868,19.785718,20.816496,21.143554,21.557222,24.540380,25.156720


============================================================
SEMAINE 1 – CHARGEMENT & DICTIONNAIRE DE DONNÉES
============================================================

In [12]:
def load_data(path: str) -> pd.DataFrame:
    """Charge le CSV et retourne un DataFrame brut."""
    df = pd.read_csv(path)
    print(f"[LOAD] {len(df)} lignes × {len(df.columns)} colonnes chargées")
    return df

def print_data_dictionary(df: pd.DataFrame):
    """Affiche le dictionnaire de données (Semaine 1)."""
    print("\n" + "="*60)
    print("DICTIONNAIRE DE DONNÉES")
    print("="*60)
    info = {
        "corps":    "VARCHAR – Corps de travaux (7 catégories)",
        "activite": "VARCHAR – Sous-catégorie d'activité (21 valeurs)",
        "produit":  "VARCHAR – Désignation du produit",
        "variete":  "VARCHAR – Variété / calibre / dimension",
        "2005-2019":"DECIMAL(10,2) – Prix moyen annuel en MAD (DH)",
    }
    for col, desc in info.items():
        print(f"  {col:<12} : {desc}")

============================================================
SEMAINE 2 – NETTOYAGE & BASE DE DONNÉES
============================================================

In [13]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Nettoie et normalise le DataFrame."""
    df = df.copy()

    # Suppression des espaces parasites dans les colonnes texte
    for col in ['corps', 'activite', 'produit', 'variete']:
        df[col] = df[col].str.strip()

    # Ajout d'un identifiant unique
    df.insert(0, 'id_materiau', range(1, len(df) + 1))

    # Vérification valeurs manquantes
    null_count = df.isnull().sum().sum()
    print(f"[CLEAN] Valeurs nulles détectées : {null_count}")

    # Doublons
    dup_count = df.duplicated(subset=['corps','activite','produit','variete']).sum()
    print(f"[CLEAN] Doublons détectés : {dup_count}")

    return df

def create_database(df: pd.DataFrame, db_path: str):
    """Crée la base SQLite normalisée en 3 tables."""
    conn = sqlite3.connect(db_path)
    cur  = conn.cursor()

    # ── Table corps ──────────────────────────────────────────
    cur.execute("DROP TABLE IF EXISTS fait_prix")
    cur.execute("DROP TABLE IF EXISTS dim_materiau")
    cur.execute("DROP TABLE IF EXISTS dim_corps")

    cur.execute("""
        CREATE TABLE dim_corps (
            id_corps  INTEGER PRIMARY KEY AUTOINCREMENT,
            nom_corps TEXT NOT NULL UNIQUE
        )
    """)

    corps_list = df['corps'].unique()
    for c in corps_list:
        cur.execute("INSERT INTO dim_corps (nom_corps) VALUES (?)", (c,))

    # ── Table matériaux ──────────────────────────────────────
    cur.execute("""
        CREATE TABLE dim_materiau (
            id_materiau INTEGER PRIMARY KEY,
            id_corps    INTEGER NOT NULL,
            activite    TEXT NOT NULL,
            produit     TEXT NOT NULL,
            variete     TEXT NOT NULL,
            FOREIGN KEY (id_corps) REFERENCES dim_corps(id_corps)
        )
    """)

    corps_map = {row[1]: row[0] for row in cur.execute("SELECT id_corps, nom_corps FROM dim_corps")}

    for _, row in df.iterrows():
        cur.execute("""
            INSERT INTO dim_materiau (id_materiau, id_corps, activite, produit, variete)
            VALUES (?, ?, ?, ?, ?)
        """, (int(row['id_materiau']), corps_map[row['corps']],
              row['activite'], row['produit'], row['variete']))

    # ── Table faits prix ─────────────────────────────────────
    cur.execute("""
        CREATE TABLE fait_prix (
            id_fait     INTEGER PRIMARY KEY AUTOINCREMENT,
            id_materiau INTEGER NOT NULL,
            annee       INTEGER NOT NULL,
            prix        DECIMAL(12,4),
            FOREIGN KEY (id_materiau) REFERENCES dim_materiau(id_materiau)
        )
    """)

    rows = []
    for _, row in df.iterrows():
        for year in YEARS:
            prix_val = row[year]
            if pd.notna(prix_val):
                rows.append((int(row['id_materiau']), int(year), float(prix_val)))

    cur.executemany(
        "INSERT INTO fait_prix (id_materiau, annee, prix) VALUES (?,?,?)", rows
    )

    conn.commit()
    conn.close()
    print(f"[DB] Base créée : {db_path}")
    print(f"[DB] {len(rows)} enregistrements insérés dans fait_prix")


============================================================
SEMAINE 3 – REQUÊTES AVANCÉES & KPI
============================================================

In [14]:
def compute_kpis(db_path: str) -> dict:
    """Calcule les KPI via SQL et retourne un dictionnaire."""
    conn = sqlite3.connect(db_path)
    kpis = {}

    # KPI 1 – Prix moyen global par corps
    kpis['prix_moyen_par_corps'] = pd.read_sql("""
        SELECT dc.nom_corps,
               ROUND(AVG(fp.prix), 2) AS prix_moyen
        FROM fait_prix fp
        JOIN dim_materiau dm ON fp.id_materiau = dm.id_materiau
        JOIN dim_corps dc    ON dm.id_corps    = dc.id_corps
        GROUP BY dc.nom_corps
        ORDER BY prix_moyen DESC
    """, conn)

    # KPI 2 – Évolution temporelle du prix moyen global
    kpis['evolution_annuelle'] = pd.read_sql("""
        SELECT annee,
               ROUND(AVG(prix), 2) AS prix_moyen_global,
               ROUND(MIN(prix), 2) AS prix_min,
               ROUND(MAX(prix), 2) AS prix_max
        FROM fait_prix
        GROUP BY annee
        ORDER BY annee
    """, conn)

    # KPI 3 – Top 5 matériaux les plus chers (2019)
    kpis['top5_chers_2019'] = pd.read_sql("""
        SELECT dm.produit, dm.variete, dc.nom_corps,
               ROUND(fp.prix, 2) AS prix_2019
        FROM fait_prix fp
        JOIN dim_materiau dm ON fp.id_materiau = dm.id_materiau
        JOIN dim_corps dc    ON dm.id_corps    = dc.id_corps
        WHERE fp.annee = 2019
        ORDER BY fp.prix DESC
        LIMIT 5
    """, conn)

    # KPI 4 – Taux de croissance 2005-2019 par corps
    kpis['croissance_par_corps'] = pd.read_sql("""
        WITH debut AS (
            SELECT dm.id_corps, ROUND(AVG(fp.prix),4) AS prix_2005
            FROM fait_prix fp
            JOIN dim_materiau dm ON fp.id_materiau = dm.id_materiau
            WHERE fp.annee = 2005
            GROUP BY dm.id_corps
        ),
        fin AS (
            SELECT dm.id_corps, ROUND(AVG(fp.prix),4) AS prix_2019
            FROM fait_prix fp
            JOIN dim_materiau dm ON fp.id_materiau = dm.id_materiau
            WHERE fp.annee = 2019
            GROUP BY dm.id_corps
        )
        SELECT dc.nom_corps,
               d.prix_2005, f.prix_2019,
               ROUND((f.prix_2019 - d.prix_2005) / d.prix_2005 * 100, 2) AS taux_croissance_pct
        FROM debut d
        JOIN fin   f  ON d.id_corps = f.id_corps
        JOIN dim_corps dc ON d.id_corps = dc.id_corps
        ORDER BY taux_croissance_pct DESC
    """, conn)

    # KPI 5 – Volatilité (écart-type) par activité
    kpis['volatilite_par_activite'] = pd.read_sql("""
        SELECT dm.activite,
               ROUND(AVG(fp.prix), 2) AS prix_moyen,
               COUNT(DISTINCT fp.annee) AS nb_annees
        FROM fait_prix fp
        JOIN dim_materiau dm ON fp.id_materiau = dm.id_materiau
        GROUP BY dm.activite
        ORDER BY prix_moyen DESC
        LIMIT 10
    """, conn)

    # KPI Opérationnel 1 – Prix ciment 2019
    kpis['ciment_2019'] = pd.read_sql("""
        SELECT dm.produit, dm.variete,
               ROUND(fp.prix, 2) AS prix_2019
        FROM fait_prix fp
        JOIN dim_materiau dm ON fp.id_materiau = dm.id_materiau
        WHERE dm.activite LIKE '%Ciment%' AND fp.annee = 2019
        ORDER BY fp.prix DESC
    """, conn)

    # KPI Opérationnel 2 – Produits dont le prix a plus que doublé
    kpis['produits_doublement_prix'] = pd.read_sql("""
        WITH debut AS (
            SELECT id_materiau, prix AS p2005
            FROM fait_prix WHERE annee = 2005
        ),
        fin AS (
            SELECT id_materiau, prix AS p2019
            FROM fait_prix WHERE annee = 2019
        )
        SELECT dm.produit, dm.variete,
               ROUND(d.p2005, 2) AS prix_2005,
               ROUND(f.p2019, 2) AS prix_2019,
               ROUND((f.p2019 - d.p2005)/d.p2005*100, 1) AS variation_pct
        FROM debut d
        JOIN fin f ON d.id_materiau = f.id_materiau
        JOIN dim_materiau dm ON d.id_materiau = dm.id_materiau
        WHERE f.p2019 > d.p2005 * 2
        ORDER BY variation_pct DESC
    """, conn)

    conn.close()
    return kpis



============================================================
SEMAINE 4 – AUTOMATISATION ETL & OPTIMISATION
============================================================

In [ ]:
def run_etl_pipeline(source_path: str, db_path: str):
    """Pipeline ETL complet : Extract → Transform → Load."""
    print("\n" + "="*60)
    print("PIPELINE ETL – AUTOMATISATION")
    print("="*60)
    print("[ETL] Étape 1/3 : Extraction des données source...")
    df_raw = load_data(source_path)

    print("[ETL] Étape 2/3 : Transformation & nettoyage...")
    df_clean = clean_data(df_raw)

    print("[ETL] Étape 3/3 : Chargement en base SQLite...")
    create_database(df_clean, db_path)

    print("[ETL] Pipeline terminé avec succès.\n")
    return df_clean

def optimize_database(db_path: str):
    """Crée des index et des vues analytiques (Semaine 4)."""
    conn = sqlite3.connect(db_path)
    cur  = conn.cursor()

    # Index sur colonnes fréquemment filtrées
    cur.execute("CREATE INDEX IF NOT EXISTS idx_prix_annee ON fait_prix(annee)")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_prix_mat   ON fait_prix(id_materiau)")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_mat_corps  ON dim_materiau(id_corps)")

    # Vue analytique 1 – vue globale prix × corps × année
    cur.execute("DROP VIEW IF EXISTS vue_prix_corps_annee")
    cur.execute("""
        CREATE VIEW vue_prix_corps_annee AS
        SELECT dc.nom_corps, fp.annee,
               ROUND(AVG(fp.prix), 2) AS prix_moyen,
               ROUND(MIN(fp.prix), 2) AS prix_min,
               ROUND(MAX(fp.prix), 2) AS prix_max,
               COUNT(*)               AS nb_produits
        FROM fait_prix fp
        JOIN dim_materiau dm ON fp.id_materiau = dm.id_materiau
        JOIN dim_corps dc    ON dm.id_corps    = dc.id_corps
        GROUP BY dc.nom_corps, fp.annee
    """)

    # Vue analytique 2 – évolution par produit
    cur.execute("DROP VIEW IF EXISTS vue_evolution_produit")
    cur.execute("""
        CREATE VIEW vue_evolution_produit AS
        SELECT dm.produit, dm.variete, dc.nom_corps,
               fp.annee,
               ROUND(fp.prix, 2) AS prix
        FROM fait_prix fp
        JOIN dim_materiau dm ON fp.id_materiau = dm.id_materiau
        JOIN dim_corps dc    ON dm.id_corps    = dc.id_corps
    """)

    # Vue analytique 3 – synthèse KPI par corps
    cur.execute("DROP VIEW IF EXISTS vue_kpi_corps")
    cur.execute("""
        CREATE VIEW vue_kpi_corps AS
        WITH debut AS (
            SELECT dm.id_corps, AVG(fp.prix) AS p2005
            FROM fait_prix fp JOIN dim_materiau dm ON fp.id_materiau = dm.id_materiau
            WHERE fp.annee = 2005 GROUP BY dm.id_corps
        ),
        fin AS (
            SELECT dm.id_corps, AVG(fp.prix) AS p2019
            FROM fait_prix fp JOIN dim_materiau dm ON fp.id_materiau = dm.id_materiau
            WHERE fp.annee = 2019 GROUP BY dm.id_corps
        )
        SELECT dc.nom_corps,
               ROUND(d.p2005, 2) AS prix_moy_2005,
               ROUND(f.p2019, 2) AS prix_moy_2019,
               ROUND((f.p2019 - d.p2005)/d.p2005*100, 2) AS hausse_pct
        FROM debut d
        JOIN fin f       ON d.id_corps = f.id_corps
        JOIN dim_corps dc ON d.id_corps = dc.id_corps
        ORDER BY hausse_pct DESC
    """)

    conn.commit()
    conn.close()
    print("[OPT] Index créés, 3 vues analytiques générées.")

def generate_sql_report(db_path: str, output_path: str):
    """Génère un script SQL documenté récapitulatif."""
    conn = sqlite3.connect(db_path)
    lines = [
        "-- ================================================",
        "-- SID – Béni Mellal-Khénifra : Matériaux (2005-2019)",
        "-- Script SQL : Structure + Requêtes Analytiques",
        "-- ================================================\n",
        "-- 1. Requête de base : liste des produits par corps",
        "SELECT dc.nom_corps, dm.activite, dm.produit, dm.variete",
        "FROM dim_materiau dm JOIN dim_corps dc ON dm.id_corps = dc.id_corps",
        "ORDER BY dc.nom_corps, dm.activite;\n",
        "-- 2. Prix moyen par corps de travaux",
        "SELECT dc.nom_corps, ROUND(AVG(fp.prix),2) AS prix_moyen",
        "FROM fait_prix fp",
        "JOIN dim_materiau dm ON fp.id_materiau = dm.id_materiau",
        "JOIN dim_corps dc ON dm.id_corps = dc.id_corps",
        "GROUP BY dc.nom_corps ORDER BY prix_moyen DESC;\n",
        "-- 3. Évolution du prix moyen global par année (KPI Temporel)",
        "SELECT annee, ROUND(AVG(prix),2) AS prix_moyen",
        "FROM fait_prix GROUP BY annee ORDER BY annee;\n",
        "-- 4. Jointure 3 tables : produits Gros œuvre avec prix 2019",
        "SELECT dm.produit, dm.variete, fp.prix",
        "FROM fait_prix fp",
        "JOIN dim_materiau dm ON fp.id_materiau = dm.id_materiau",
        "JOIN dim_corps dc    ON dm.id_corps    = dc.id_corps",
        "WHERE dc.nom_corps = 'Gros œuvre' AND fp.annee = 2019",
        "ORDER BY fp.prix DESC;\n",
        "-- 5. Taux de croissance 2005-2019 par corps (CTE optimisé)",
        "WITH debut AS (",
        "    SELECT dm.id_corps, AVG(fp.prix) AS p2005",
        "    FROM fait_prix fp JOIN dim_materiau dm ON fp.id_materiau = dm.id_materiau",
        "    WHERE fp.annee = 2005 GROUP BY dm.id_corps",
        "),",
        "fin AS (",
        "    SELECT dm.id_corps, AVG(fp.prix) AS p2019",
        "    FROM fait_prix fp JOIN dim_materiau dm ON fp.id_materiau = dm.id_materiau",
        "    WHERE fp.annee = 2019 GROUP BY dm.id_corps",
        ")",
        "SELECT dc.nom_corps,",
        "       ROUND(d.p2005, 2) AS prix_2005,",
        "       ROUND(f.p2019, 2) AS prix_2019,",
        "       ROUND((f.p2019 - d.p2005)/d.p2005*100, 2) AS taux_croissance_pct",
        "FROM debut d JOIN fin f ON d.id_corps = f.id_corps",
        "JOIN dim_corps dc ON d.id_corps = dc.id_corps",
        "ORDER BY taux_croissance_pct DESC;\n",
        "-- 6. HAVING : corps dont le prix moyen 2019 dépasse 200 MAD",
        "SELECT dc.nom_corps, ROUND(AVG(fp.prix),2) AS prix_moy",
        "FROM fait_prix fp",
        "JOIN dim_materiau dm ON fp.id_materiau = dm.id_materiau",
        "JOIN dim_corps dc ON dm.id_corps = dc.id_corps",
        "WHERE fp.annee = 2019",
        "GROUP BY dc.nom_corps HAVING prix_moy > 200",
        "ORDER BY prix_moy DESC;\n",
        "-- 7. CASE WHEN : catégorisation des prix 2019",
        "SELECT dm.produit,",
        "       ROUND(fp.prix,2) AS prix_2019,",
        "       CASE",
        "           WHEN fp.prix < 10   THEN 'Très bas'",
        "           WHEN fp.prix < 50   THEN 'Bas'",
        "           WHEN fp.prix < 200  THEN 'Moyen'",
        "           WHEN fp.prix < 1000 THEN 'Élevé'",
        "           ELSE 'Très élevé'",
        "       END AS categorie_prix",
        "FROM fait_prix fp",
        "JOIN dim_materiau dm ON fp.id_materiau = dm.id_materiau",
        "WHERE fp.annee = 2019 ORDER BY fp.prix DESC;\n",
        "-- 8. Sous-requête : produits au-dessus du prix moyen général 2019",
        "SELECT dm.produit, dm.variete, ROUND(fp.prix,2) AS prix",
        "FROM fait_prix fp",
        "JOIN dim_materiau dm ON fp.id_materiau = dm.id_materiau",
        "WHERE fp.annee = 2019",
        "  AND fp.prix > (SELECT AVG(prix) FROM fait_prix WHERE annee = 2019)",
        "ORDER BY fp.prix DESC;\n",
        "-- 9. Procédure stockée simulée : détection anomalies prix nul",
        "-- (SQLite ne supporte pas nativement les procédures, logique équivalente)",
        "SELECT dm.produit, fp.annee, fp.prix",
        "FROM fait_prix fp JOIN dim_materiau dm ON fp.id_materiau = dm.id_materiau",
        "WHERE fp.prix IS NULL OR fp.prix <= 0;\n",
        "-- 10. Vue récapitulative KPI globaux",
        "SELECT * FROM vue_kpi_corps;",
    ]
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    conn.close()
    print(f"[SQL] Script SQL exporté : {output_path}")

============================================================
MAIN
============================================================

In [ ]:
if __name__ == "__main__":
    # os.chdir(os.path.dirname(os.path.abspath(__file__)))

    print_data_dictionary(pd.read_csv(
        DATA_PATH, nrows=1
    ))

    # ETL complet
    df_clean = run_etl_pipeline(
        DATA_PATH,
        DB_PATH
    )

    # Optimisation
    optimize_database(DB_PATH)

    # Calcul des KPI
    kpis = compute_kpis(DB_PATH)

    print("\n" + "="*60)
    print("KPI 1 – PRIX MOYEN PAR CORPS DE TRAVAUX")
    print("="*60)
    print(kpis['prix_moyen_par_corps'].to_string(index=False))

    print("\n" + "="*60)
    print("KPI 2 – ÉVOLUTION ANNUELLE DU PRIX MOYEN")
    print("="*60)
    print(kpis['evolution_annuelle'].to_string(index=False))

    print("\n" + "="*60)
    print("KPI 3 – TOP 5 MATÉRIAUX LES PLUS CHERS EN 2019")
    print("="*60)
    print(kpis['top5_chers_2019'].to_string(index=False))

    print("\n" + "="*60)
    print("KPI 4 – TAUX DE CROISSANCE 2005-2019 PAR CORPS")
    print("="*60)
    print(kpis['croissance_par_corps'].to_string(index=False))

    print("\n" + "="*60)
    print("KPI 5 – VOLATILITÉ PAR ACTIVITÉ (Top 10)")
    print("="*60)
    print(kpis['volatilite_par_activite'].to_string(index=False))

    print("\n" + "="*60)
    print("KPI OP.1 – PRIX CIMENT 2019")
    print("="*60)
    print(kpis['ciment_2019'].to_string(index=False))

    print("\n" + "="*60)
    print("KPI OP.2 – PRODUITS DONT LE PRIX A PLUS QUE DOUBLÉ")
    print("="*60)
    print(kpis['produits_doublement_prix'].to_string(index=False))

    # Export SQL
    generate_sql_report(DB_PATH, "sid_requetes.sql")

    print("\n[DONE] Tous les livrables Python ont été générés.")



DICTIONNAIRE DE DONNÉES
  corps        : VARCHAR – Corps de travaux (7 catégories)
  activite     : VARCHAR – Sous-catégorie d'activité (21 valeurs)
  produit      : VARCHAR – Désignation du produit
  variete      : VARCHAR – Variété / calibre / dimension
  2005-2019    : DECIMAL(10,2) – Prix moyen annuel en MAD (DH)

PIPELINE ETL – AUTOMATISATION
[ETL] Étape 1/3 : Extraction des données source...
[LOAD] 84 lignes × 19 colonnes chargées
[ETL] Étape 2/3 : Transformation & nettoyage...
[CLEAN] Valeurs nulles détectées : 0
[CLEAN] Doublons détectés : 0
[ETL] Étape 3/3 : Chargement en base SQLite...
[DB] Base créée : materiaux_beni_mellal.db
[DB] 1260 enregistrements insérés dans fait_prix
[ETL] Pipeline terminé avec succès.

[OPT] Index créés, 3 vues analytiques générées.

KPI 1 – PRIX MOYEN PAR CORPS DE TRAVAUX
                  nom_corps  prix_moyen
   Menuiserie Quincaillerie     2657.04
                 Revêtement      239.08
        Plomberie Sanitaire      220.82
                 G